<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# Procesamiento de lenguaje natural
# Desafio 4, LSTM Bot QA

In [102]:
import sys

if 'google.colab' in sys.modules:
    print("Entorno Colab detectado. Asegurando instalación de librerias...")
    !pip install -q torchinfo

In [132]:
import os
import re
import json
import requests

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

import urllib.request
import gzip
import shutil
import logging
from pathlib import Path
import pickle

from torch_helpers import Tokenizer

In [127]:
# Configurar el dispositivo para PyTorch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"El dispositivo seleccionado es: {device}")

if device.type == 'cuda':
    print(f"Nombre de la GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Se está utilizando la CPU.")

total_cores = os.cpu_count()
optimal_workers = min(total_cores - 2, 8)

print(f"Hardware detectado: {total_cores} núcleos.")
print(f"Asignando {optimal_workers} workers para optimizar la transferencia a la GPU.")

El dispositivo seleccionado es: cuda
Nombre de la GPU: NVIDIA GeForce RTX 5070 Ti
Hardware detectado: 24 núcleos.
Asignando 8 workers para optimizar la transferencia a la GPU.


In [105]:
# Descarga la libreria torch_helpers.py provista en el curso

url = "https://raw.githubusercontent.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/main/scripts/torch_helpers.py"
filename = "torch_helpers.py"

if not os.path.exists(filename):
    print(f"Archivo '{filename}' no encontrado. Descargando...")
    try:
        response = requests.get(url)
        response.raise_for_status()  # Verifica si la descarga fue exitosa
        with open(filename, "wb") as f:
            f.write(response.content)
        print("Descarga completada con éxito.")
    except Exception as e:
        print(f"Error al descargar: {e}")
else:
    print(f"El archivo '{filename}' ya existe en el directorio.")

El archivo 'torch_helpers.py' ya existe en el directorio.


### 1 - Datos
El objetivo es utilizar datos disponibles del challenge ConvAI2 (Conversational Intelligence Challenge 2) de conversaciones en inglés. Se construirá un BOT para responder a preguntas del usuario (QA).\
[LINK](http://convai.io/data/)

In [106]:
# Descarga el contenido del dataset desde github

url = "https://raw.githubusercontent.com/mgonzalez738/desafios_procesamiento_lenguaje_natural/main/data_volunteers.json"
filename = "data_volunteers.json"

if not os.path.exists(filename):
    print(f"Archivo '{filename}' no encontrado. Descargando...")
    try:
        response = requests.get(url)
        response.raise_for_status()  # Verifica si la descarga fue exitosa
        with open(filename, "wb") as f:
            f.write(response.content)
        print("Descarga completada con éxito.")
    except Exception as e:
        print(f"Error al descargar: {e}")
else:
    print(f"El archivo '{filename}' ya existe en el directorio.")

El archivo 'data_volunteers.json' ya existe en el directorio.


In [107]:
# Carga el archivo del dataset

if os.path.exists(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f"Archivo cargado exitosamente. Total de conjuntos de dialogos: {len(data)}")
else:
    print("Error: No se encontró el archivo data_volunteers.json en la ruta especificada.")

# Imprime el primer objeto del dataset de forma legible

if len(data) > 0:
    print(json.dumps(data[8], indent=4))
else:
    print("El dataset está vacío.")

Archivo cargado exitosamente. Total de conjuntos de dialogos: 1111
{
    "dialog": [
        {
            "id": 0,
            "sender": "participant1",
            "text": "Hi!",
            "evaluation_score": null,
            "sender_class": "Human"
        },
        {
            "id": 1,
            "sender": "participant2",
            "text": "hi how are you today? i am doing well. ",
            "evaluation_score": null,
            "sender_class": "Bot"
        },
        {
            "id": 2,
            "sender": "participant1",
            "text": "Spent a day in my studio.",
            "evaluation_score": null,
            "sender_class": "Human"
        },
        {
            "id": 3,
            "sender": "participant2",
            "text": "that sounds fun. what kind of studio? ",
            "evaluation_score": null,
            "sender_class": "Bot"
        },
        {
            "id": 4,
            "sender": "participant1",
            "text": "Recording st

In [108]:
# Obtiene los pares de mensajes Human -> Bot y los preprocesa

# Función de limpieza de texto
# 1 - Convertir a minúsculas
# 2 - Expandir contracciones comunes (ej. "don't" -> "do not")
# 3 - Eliminar caracteres no alfanuméricos (excepto espacios)
def clean_text(txt):
    txt = txt.lower()
    txt = txt.replace("'d", " had")
    txt = txt.replace("'s", " is")
    txt = txt.replace("'m", " am")
    txt = txt.replace("don't", " do not")
    txt = txt.replace("can't", " cannot")
    txt = txt.replace("n't", " not")
    txt = re.sub(r"([?.!,])", r" \1 ", txt)    # Agrega espacios alrededor de signos de puntuación comunes
    txt = re.sub(r'[^a-z0-9?.!,\s]', ' ', txt) # Permite signos de puntuación comunes
    return " ".join(txt.split())

# Inicializa las listas para almacenar los pares de mensajes
input_sentences = []
output_sentences = []
output_sentences_inputs = []
max_words = 30

discarded_empty_message = 0
discarded_short_dialogs = 0
discarded_consecutive_turns = 0
discarded_by_length = 0

# Recorre cada diálogo en el dataset

for line in data:
    dialog = line.get('dialog', [])
    
    # Si el diálogo tiene menos de 2 mensajes lo ignora
    if len(dialog) < 2:
        discarded_short_dialogs += 1
        continue

    # Recorre el diálogo buscando pares Human -> Bot
    for i in range(len(dialog) - 1):
        msg_current = dialog[i]
        msg_next = dialog[i+1]
        
        # Validación solo pares Human -> Bot
        if msg_current.get('sender_class') == 'Human' and msg_next.get('sender_class') == 'Bot':
            
            raw_in = msg_current['text']
            raw_out = msg_next['text']
            
            chat_in = clean_text(raw_in)
            chat_out = clean_text(raw_out)
            
            if not chat_in or not chat_out:
                discarded_empty_message += 1
                continue
                
            if len(chat_in.split()) > max_words or len(chat_out.split()) > max_words:
                discarded_by_length += 1
                continue
                
            # Guarda el par de mensajes preprocesados
            input_sentences.append(chat_in)
            output_sentences_inputs.append(f'<sos> {chat_out}')
            output_sentences.append(f'{chat_out} <eos>')

        # Si el humano manda dos mensajes seguidos, o el bot dos seguidos, se ignoran    
        elif msg_current.get('sender_class') == msg_next.get('sender_class'):
            discarded_consecutive_turns += 1

print(f"Pares útiles extraídos: {len(input_sentences)}")
print(f"Descartados por solo tener un mensaje: {discarded_short_dialogs}")
print(f"Descartados por tener emisor consecutivo: {discarded_consecutive_turns}")
print(f"Descartados por exceder {max_words} palabras: {discarded_by_length}")
print(f"Descartados por tener algun mensajes vacío: {discarded_empty_message}")

print("\n Ejemplo de pares procesados:")

for i in range(4):
    print(f"  [Encoder Input]  : {input_sentences[i]}")
    print(f"  [Decoder Input]  : {output_sentences_inputs[i]}")
    print(f"  [Decoder Target] : {output_sentences[i]}")

Pares útiles extraídos: 6248
Descartados por solo tener un mensaje: 220
Descartados por tener emisor consecutivo: 1197
Descartados por exceder 30 palabras: 105
Descartados por tener algun mensajes vacío: 45

 Ejemplo de pares procesados:
  [Encoder Input]  : hello !
  [Decoder Input]  : <sos> hi ! how are you ?
  [Decoder Target] : hi ! how are you ? <eos>
  [Encoder Input]  : not bad ! and you ?
  [Decoder Input]  : <sos> i am doing well . just got engaged to my high school sweetheart .
  [Decoder Target] : i am doing well . just got engaged to my high school sweetheart . <eos>
  [Encoder Input]  : wowowowow ! congratulations ! is she pretty ?
  [Decoder Input]  : <sos> she is pretty cute . she invited me to dinner tonight .
  [Decoder Target] : she is pretty cute . she invited me to dinner tonight . <eos>
  [Encoder Input]  : cool ! have a good time you both ! and what is your hobby ?
  [Decoder Input]  : <sos> i love music ! i love taylor swift .
  [Decoder Target] : i love music ! 

### 2 - Preprocesamiento
Realizar el preprocesamiento necesario para obtener:
- word2idx_inputs, max_input_len
- word2idx_outputs, max_out_len, num_words_output
- encoder_input_sequences, decoder_output_sequences, decoder_targets

In [110]:
# Limita el vocabulario a las MAX_VOCAB_SIZE palabras más frecuentes
MAX_VOCAB_SIZE = 10000

In [111]:
# Inicializa el tokenizador para las preguntas del usuario 
# Evita filtrar signos de puntuacion comunes para preservar el significado de las frases (, . ? !)
input_tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, filters='"<>¿¡#$%&()*+-/:;=@[\\]^_`{|}~\t\n')

# Ajusta el tokenizador mediante una lista de textos.
# Cuenta las palabras y asigna un índice a cada una, ordenándolas por frecuencia (mas usadas a menos usadas).
# El ID 0 se reserva para el padding.
input_tokenizer.fit_on_texts(input_sentences)

# Convierte cada texto en una secuencia de enteros, donde cada entero representa el índice de una palabra en el vocabulario.
input_integer_seq = input_tokenizer.texts_to_sequences(input_sentences)

# Imprime el vocabulario y la longitud de la sentencia de entrada más larga

word2idx_inputs = input_tokenizer.word_index
print("Palabras en el vocabulario de entrada:", len(word2idx_inputs))
items_word2idx_inputs = list(word2idx_inputs.items())
print(f"\nLas 5 palabras más frecuentes del vocabulario:")
for palabra, token_id in items_word2idx_inputs[:5]:
    print(f"Token ID: {token_id:4} | Palabra: '{palabra}'")

max_input_len = max(len(sen) for sen in input_integer_seq)
print("\nSentencia de entrada más larga:", max_input_len)
for i in range(min(2, len(input_sentences))):
    text = input_sentences[i]
    sequence = input_integer_seq[i]
    print(f"\nEjemplo {i+1}:")
    print(f"Texto Original : '{text}'")
    print(f"Secuencia IDs  : {sequence}")

Palabras en el vocabulario de entrada: 3123

Las 5 palabras más frecuentes del vocabulario:
Token ID:    1 | Palabra: 'i'
Token ID:    2 | Palabra: '?'
Token ID:    3 | Palabra: 'you'
Token ID:    4 | Palabra: '.'
Token ID:    5 | Palabra: 'do'

Sentencia de entrada más larga: 30

Ejemplo 1:
Texto Original : 'hello !'
Secuencia IDs  : [49, 17]

Ejemplo 2:
Texto Original : 'not bad ! and you ?'
Secuencia IDs  : [14, 217, 17, 15, 3, 2]


In [112]:
# Inicializa el tokenizador para las respuestas del bot

# Evita filtrar signos de puntuacion comunes para preservar el significado de las frases (, . ? !) y < > para <sos> y <eos>
output_tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, filters='"¿¡#$%&()*+-/:;=@[\\]^_`{|}~\t\n')

# Ajusta el tokenizador mediante una lista de textos agregando los tokens especiales (solo falta <sos>).
output_tokenizer.fit_on_texts(["<sos>"] + output_sentences)

output_integer_seq = output_tokenizer.texts_to_sequences(output_sentences)
output_input_integer_seq = output_tokenizer.texts_to_sequences(output_sentences_inputs)

word2idx_outputs = output_tokenizer.word_index
print("Palabras en el vocabulario de salida:", len(word2idx_outputs))
items_word2idx_outputs = list(word2idx_outputs.items())
print(f"\nLas 5 palabras más frecuentes del vocabulario de salida:")
for palabra, token_id in items_word2idx_outputs[:5]:
    print(f"Token ID: {token_id:4} | Palabra: '{palabra}'")

num_words_output = min(len(word2idx_outputs) + 1, MAX_VOCAB_SIZE) # Se suma 1 por el primer <sos>
max_out_len = max(len(sen) for sen in output_integer_seq)
print("\nSentencia de salida más larga:", max_out_len)

for i in range(min(1, len(output_sentences))):
    text = output_sentences[i]
    sequence = output_integer_seq[i]
    print(f"\nEjemplo {i+1}:")
    print(f"Texto Original : '{text}'")
    print(f"Secuencia IDs  : {sequence}")

for i in range(min(1, len(output_sentences_inputs))):
    text = output_sentences_inputs[i]
    sequence = output_input_integer_seq[i]
    print(f"Texto Original : '{text}'")
    print(f"Secuencia IDs  : {sequence}")


Palabras en el vocabulario de salida: 1780

Las 5 palabras más frecuentes del vocabulario de salida:
Token ID:    1 | Palabra: '<eos>'
Token ID:    2 | Palabra: '.'
Token ID:    3 | Palabra: 'i'
Token ID:    4 | Palabra: '?'
Token ID:    5 | Palabra: 'you'

Sentencia de salida más larga: 31

Ejemplo 1:
Texto Original : 'hi ! how are you ? <eos>'
Secuencia IDs  : [43, 30, 17, 14, 5, 4, 1]
Texto Original : '<sos> hi ! how are you ?'
Secuencia IDs  : [1118, 43, 30, 17, 14, 5, 4]


In [ ]:
# Se crea esta función de padding por incompatibilidad de la función en la libreria con numpy >= 2.0.

def pad_sequences(sequences, maxlen=None, padding='pre', value=0):

    num_samples = len(sequences)
    
    # Si no se especifica maxlen, toma la de la secuencia más larga
    if maxlen is None:
        maxlen = max(len(s) for s in sequences)

    # Crea la matriz rectangular llena del valor de padding (0 por defecto)
    x = np.full((num_samples, maxlen), value, dtype='int32')

    for idx, s in enumerate(sequences):
        if not len(s):
            continue
            
        # Truncamiento (si la frase es más larga que maxlen, corta el final)
        trunc = s[:maxlen]
        
        # Padding 
        if padding == 'post':
            x[idx, :len(trunc)] = trunc
        else: # 'pre' (por defecto)
            x[idx, -len(trunc):] = trunc
            
    return x

In [124]:
# Genera las matrices de secuencias de enteros con padding para encoder y decoder

print("Cantidad de rows del dataset:", len(input_integer_seq))

encoder_input_sequences = pad_sequences(input_integer_seq, maxlen=max_input_len)
print("encoder_input_sequences shape:", encoder_input_sequences.shape)

decoder_input_sequences = pad_sequences(output_input_integer_seq, maxlen=max_out_len, padding='post')
print("decoder_input_sequences shape:", decoder_input_sequences.shape)

decoder_output_sequences = pad_sequences(output_integer_seq, maxlen=max_out_len, padding='post')
print("decoder_output_sequences shape:", decoder_output_sequences.shape)

print("\nEjemplos de secuencias con padding:")
samples_qty = 1

for i in range(samples_qty):
    print(f"  Encoder Input (pre-padded) :\n  {encoder_input_sequences[i]}")
    print(f"  Decoder Input (post-padded) :\n  {decoder_input_sequences[i]}")
    print(f"  Decoder Output (post-padded) :\n  {decoder_output_sequences[i]}")

Cantidad de rows del dataset: 6248
encoder_input_sequences shape: (6248, 30)
decoder_input_sequences shape: (6248, 31)
decoder_output_sequences shape: (6248, 31)

Ejemplos de secuencias con padding:
  Encoder Input (pre-padded) :
  [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0 49 17]
  Decoder Input (post-padded) :
  [1118   43   30   17   14    5    4    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0]
  Decoder Output (post-padded) :
  [43 30 17 14  5  4  1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0]


In [128]:
# Define la clase Dataset personalizada para manejar los datos de entrenamiento

class Data(Dataset):
    def __init__(self, encoder_input, decoder_input, decoder_output, num_classes):
        # Convierte los arrays de numpy a tensores. 
        self.encoder_inputs = torch.from_numpy(encoder_input.astype(np.int32))
        self.decoder_inputs = torch.from_numpy(decoder_input.astype(np.int32))
        
        # Transforma a One-Hot Encoding la salida (Target)
        self.decoder_outputs = F.one_hot(
            torch.from_numpy(decoder_output).to(torch.int64), 
            num_classes=num_classes
        ).float()

        self.len = self.encoder_inputs.shape[0]

    def __getitem__(self, index):
        # Retorna el pack necesario para un paso de entrenamiento Seq2Seq
        return self.encoder_inputs[index], self.decoder_inputs[index], self.decoder_outputs[index]

    def __len__(self):
        return self.len


# Instancia el dataset personalizado con las secuencias de entrada, de salida y la cantidad de clases
data_set = Data(encoder_input_sequences, decoder_input_sequences, decoder_output_sequences, num_classes=num_words_output)

# Guarda las dimensiones para la arquitectura del modelo
encoder_seq_len = data_set.encoder_inputs.shape[1]
decoder_seq_len = data_set.decoder_inputs.shape[1]
output_dim = data_set.decoder_outputs.shape[2]

print(f"Longitud secuencia Encoder: {encoder_seq_len}")
print(f"Longitud secuencia Decoder: {decoder_seq_len}")
print(f"Dimensión de salida (Vocab): {output_dim}")

# Separacion de entrenamiento y validacion

torch.manual_seed(42) 
valid_set_size = int(len(data_set) * 0.2)
train_set_size = len(data_set) - valid_set_size

train_set = Subset(data_set, range(train_set_size))
valid_set = Subset(data_set, range(train_set_size, len(data_set)))

print("Tamaño del conjunto de entrenamiento:", len(train_set))
print("Tamaño del conjunto de validacion:", len(valid_set))

# Crea los dataloaders para entrenamiento y validación

train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=optimal_workers, pin_memory=True)
valid_loader = DataLoader(valid_set, batch_size=128, shuffle=False, num_workers=optimal_workers, pin_memory=True)

Longitud secuencia Encoder: 30
Longitud secuencia Decoder: 31
Dimensión de salida (Vocab): 1781
Tamaño del conjunto de entrenamiento: 4999
Tamaño del conjunto de validacion: 1249


### 3 - Preparar los embeddings
Utilizar los embeddings de Glove o FastText para transformar los tokens de entrada en vectores

In [131]:
# Descarga y extraccion de FastText para cargar los embeddings preentrenados

gz_file = 'cc.en.300.vec.gz'
vec_file = 'cc.en.300.vec'
pkl_file = 'fasttext.pkl'

# URL oficial de los vectores en inglés de FastText (Facebook Research)
fasttext_url = 'https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.vec.gz'

# Solo descarga si no existe ni el .pkl ni el .vec
if not os.path.exists(pkl_file) and not os.path.exists(vec_file):
    if not os.path.exists(gz_file):
        print(f"Descargando FastText (1.2 GB) desde {fasttext_url}...")
        urllib.request.urlretrieve(fasttext_url, gz_file)
        print("Descarga completada.")
    
    print(f"Descomprimiendo {gz_file}...")
    with gzip.open(gz_file, 'rb') as f_in:
        with open(vec_file, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
    print("Descompresión completada.")
else:
    print("Los archivos de FastText ya están presentes o procesados.")

Los archivos de FastText ya están presentes o procesados.


In [133]:
# Clases de gestion de embeddings preentrenados de FastText

class WordsEmbeddings(object):
    logger = logging.getLogger(__name__)

    def __init__(self):
        words_embedding_pkl = Path(self.PKL_PATH)

        # Si no existe el pickle, lo crea parseando el archivo de texto 
        if not words_embedding_pkl.is_file():
            words_embedding_txt = Path(self.WORD_TO_VEC_MODEL_TXT_PATH)
            assert words_embedding_txt.is_file(), f'El archivo {self.WORD_TO_VEC_MODEL_TXT_PATH} no existe.'
            embeddings = self.convert_model_to_pickle()
        else:
            embeddings = self.load_model_from_pickle()
            
        self.embeddings = embeddings
        index = np.arange(self.embeddings.shape[0])
        
        # Diccionarios para traducir de palabra a IDX del embedding
        self.word2idx = dict(zip(self.embeddings['word'], index))
        self.idx2word = dict(zip(index, self.embeddings['word']))

    def get_words_embeddings(self, words):
        words_idxs = self.words2idxs(words)
        return self.embeddings[words_idxs]['embedding']

    def words2idxs(self, words):
        return np.array([self.word2idx.get(word, -1) for word in words])

    def idxs2words(self, idxs):
        return np.array([self.idx2word.get(idx, '-1') for idx in idxs])

    def load_model_from_pickle(self):
        print(f"Cargando embeddings desde la caché rápida: {self.PKL_PATH}")
        max_bytes = 2**28 - 1 # 256MB chunks
        bytes_in = bytearray(0)
        input_size = os.path.getsize(self.PKL_PATH)
        with open(self.PKL_PATH, 'rb') as f_in:
            for _ in range(0, input_size, max_bytes):
                bytes_in += f_in.read(max_bytes)
        embeddings = pickle.loads(bytes_in)
        return embeddings

    def convert_model_to_pickle(self):
        print(f"Parseando el archivo de texto {self.WORD_TO_VEC_MODEL_TXT_PATH} por primera vez...")
        print("Esto consumirá bastante RAM y tardará un rato. Al finalizar se creará un .pkl para que la próxima vez sea instantáneo.")
        
        structure = [('word', np.dtype('U' + str(self.WORD_MAX_SIZE))),
                     ('embedding', np.float32, (self.N_FEATURES,))]
        structure = np.dtype(structure)
        
        with open(self.WORD_TO_VEC_MODEL_TXT_PATH, encoding="utf8") as words_embeddings_txt:
            embeddings_gen = (
                (line.split()[0], line.split()[1:]) for line in words_embeddings_txt
                if len(line.split()[1:]) == self.N_FEATURES
            )
            embeddings = np.fromiter(embeddings_gen, structure)
            
        # Agregamos el "null embedding" (vector de ceros) al final para palabras no encontradas
        null_embedding = np.array(
            [('null_embedding', np.zeros((self.N_FEATURES,), dtype=np.float32))],
            dtype=structure
        )
        embeddings = np.concatenate([embeddings, null_embedding])
        
        print(f"Guardando caché en {self.PKL_PATH}...")
        max_bytes = 2**28 - 1 
        bytes_out = pickle.dumps(embeddings, protocol=pickle.HIGHEST_PROTOCOL)
        with open(self.PKL_PATH, 'wb') as f_out:
            for idx in range(0, len(bytes_out), max_bytes):
                f_out.write(bytes_out[idx:idx+max_bytes])
        return embeddings


class FasttextEmbeddings(WordsEmbeddings):
    WORD_TO_VEC_MODEL_TXT_PATH = vec_file
    PKL_PATH = pkl_file
    N_FEATURES = 300 # 
    WORD_MAX_SIZE = 60

In [ ]:
# Creación de la matriz de embeddings

print('\nIniciando carga de FastText a memoria RAM...')
model_embeddings = FasttextEmbeddings() 

print('Cruzando vocabulario del Tokenizer con FastText para armar la Embedding Matrix...')
embed_dim = model_embeddings.N_FEATURES
words_not_found = []

# Determinar el tamaño de la matriz (max vocabulario o vocabulario real + 1 por padding)
nb_words = min(MAX_VOCAB_SIZE, len(word2idx_inputs) + 1)
embedding_matrix = np.zeros((nb_words, embed_dim), dtype=np.float32)

for word, i in word2idx_inputs.items():
    if i >= nb_words:
        continue
        
    # Pide el vector. Si no existe, devuelve el array de ceros (null_embedding)
    embedding_vector = model_embeddings.get_words_embeddings([word])[0]
    
    if np.any(embedding_vector): 
        embedding_matrix[i] = embedding_vector
    else:
        words_not_found.append(word)

print(f"\nForma de la Matriz de Embeddings : {embedding_matrix.shape}")
print(f"Dimensiones por palabra          : {embed_dim} (FastText)")
print(f"Palabras no encontradas (OOV)    : {len(words_not_found)}")
print(f"Porcentaje OOV                   : {(len(words_not_found) / nb_words) * 100:.2f}%")


Iniciando carga de FastText a memoria RAM...
Cargando embeddings desde la caché rápida: fasttext.pkl
Cruzando vocabulario del Tokenizer con FastText para armar la Embedding Matrix...

Forma de la Matriz de Embeddings : (3124, 300)
Dimensiones por palabra          : 300 (FastText)
Palabras no encontradas (OOV)    : 170
Porcentaje OOV                   : 5.44%


### 4 - Entrenar el modelo
Entrenar un modelo basado en el esquema encoder-decoder utilizando los datos generados en los puntos anteriores. Utilce como referencias los ejemplos vistos en clase.

### 5 - Inferencia
Experimentar el funcionamiento de su modelo. Recuerde que debe realizar la inferencia de los modelos por separado de encoder y decoder.